# Xay Dung & Chuan Hoa Bang Quan Sat Chat Luong Du Lieu: dq_results

Notebook nay tap trung xay dung bang quan sat va giam sat chat luong du lieu **`dq_results`** (Data Quality Observability) trong kien truc Silver Layer:
- Ghi nhan ket qua kiem tra chat luong du lieu theo tung batch, theo tung nguon thu thap va theo tung bang trong he thong.
- Ap dung bo tieu chuan 6 chieu chat luong du lieu (DAMA / Great Expectations): Completeness (Day du), Uniqueness (Duy nhat), Validity (Hop le), Integrity (Toan ven lien ket), Business Logic (Logic nghiep vu), va Freshness (Do tuoi moi).
- Luu tru chi tiet so ban ghi dat (`passed_records`), so ban ghi loi (`failed_records`), ty le dat (`pass_rate`), va tom tat loi duoi dang JSON (`error_summary`).
- Kiem tra nghiem ngat rang buoc DDL PostgreSQL: khoa chinh `dq_id`, check constraint `chk_dq_severity`, check constraint `chk_dq_pass_rate`, va khoa ngoai tham chieu `job_sources(source_id)`.

In [1]:
import os
import json
from collections import Counter
from datetime import datetime
import pandas as pd
import numpy as np

# Cau hinh hien thi pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
print('Moi truong xu ly bang dq_results da san sang!')

Moi truong xu ly bang dq_results da san sang!


## 1. Nap Du Lieu Cac Bang Silver & Thiet Lap Khung Kiem Dinh (DQ Framework)

Nap du lieu tu cac tang Master Bronze da duoc chuan hoa de tien hanh kiem tra toan dien tren 8 bang Silver da xay dung.

In [2]:
listings_path = os.path.abspath('../../data/bronze/bronze_listings_master.json')
details_path = os.path.abspath('../../data/bronze/bronze_details_valid.json')

with open(listings_path, 'r', encoding='utf-8') as f:
    bronze_listings = json.load(f)
with open(details_path, 'r', encoding='utf-8') as f:
    bronze_details = json.load(f)

df_listings = pd.DataFrame(bronze_listings)
df_details = pd.DataFrame(bronze_details)

df_merged = pd.merge(
    df_listings,
    df_details,
    on=['source', 'source_job_id'],
    how='inner',
    suffixes=('_list', '_detail')
)
df_merged['job_id'] = df_merged.index + 1

source_map = {'topcv': 1, 'topdev': 2, 'vietnamworks': 3}
df_merged['source_id'] = df_merged['source'].map(source_map)

print(f'Tong so ban ghi san sang de kiem dinh chat luong: {len(df_merged):,} ban ghi')
print(f'Phan bo theo nguon: {Counter(df_merged["source"])}')

Tong so ban ghi san sang de kiem dinh chat luong: 2,327 ban ghi
Phan bo theo nguon: Counter({'topcv': 1161, 'vietnamworks': 1046, 'topdev': 120})


## 2. Thiet Ke & Thuc Thi Bo Quy Tac Kiem Dinh Chat Luong Du Lieu (Data Quality Rules)

Thiet ke bo 14 quy tac kiem dinh bao phu toan bo he thong Silver Layer voi cac cap do nghiem trong: `critical`, `error`, `warning`, `info`.

In [3]:
now_iso = datetime.now().isoformat()
batch_id = 'batch_silver_20261013_01'
dq_records = []
dq_counter = 1

def add_dq_result(table_name, rule_name, rule_type, severity, total, passed, source_id=None, sample_errors=None):
    global dq_counter
    total_int = int(total)
    passed_int = int(passed)
    failed_int = int(total_int - passed_int)
    rate = round((passed_int / total_int * 100.0), 2) if total_int > 0 else 100.0
    summary = {
        'failed_count': failed_int,
        'sample_violations': sample_errors or []
    }
    dq_records.append({
        'dq_id': int(dq_counter),
        'batch_id': str(batch_id),
        'source_id': int(source_id) if source_id is not None else None,
        'table_name': str(table_name),
        'rule_name': str(rule_name),
        'rule_type': str(rule_type),
        'severity': str(severity),
        'total_records': total_int,
        'passed_records': passed_int,
        'failed_records': failed_int,
        'pass_rate': rate,
        'error_summary': json.dumps(summary, ensure_ascii=False),
        'checked_at': now_iso
    })
    dq_counter += 1

# --- THUC THI KIEM TRA CAC QUY TAC --- #

# Quy tac 1: Completeness - Chuc danh cong viec khong duoc rong
valid_titles = df_merged['title_raw_list'].fillna('').astype(str).str.strip().str.len() >= 2
add_dq_result('jobs', 'chk_title_not_empty', 'completeness', 'critical', len(df_merged), valid_titles.sum())

# Quy tac 2: Completeness - Ten cong ty khong duoc rong
valid_comps = df_merged['company_name_raw_list'].fillna('').astype(str).str.strip().str.len() >= 2
add_dq_result('companies', 'chk_company_name_not_empty', 'completeness', 'critical', len(df_merged), valid_comps.sum())

# Quy tac 3: Completeness - Noi dung mo ta / yeu cau khong duoc rong
valid_descs = (df_merged['responsibilities_raw'].fillna('').astype(str).str.strip().str.len() > 10) | (df_merged['requirements_raw'].fillna('').astype(str).str.strip().str.len() > 10)
add_dq_result('job_descriptions', 'chk_jd_content_not_empty', 'completeness', 'critical', len(df_merged), valid_descs.sum())

# Quy tac 4: Validity - URL tin tuyen dung phai hop le (chua http/https)
valid_urls = df_merged['job_url_list'].fillna('').astype(str).str.startswith(('http://', 'https://'))
add_dq_result('job_source_records', 'chk_valid_job_url_format', 'validity', 'error', len(df_merged), valid_urls.sum())

# Quy tac 5: Uniqueness - Cap (source_id, source_job_id) phai duy nhat
unique_source_recs = len(df_merged.drop_duplicates(subset=['source', 'source_job_id']))
add_dq_result('job_source_records', 'chk_unique_source_job_id', 'uniqueness', 'critical', len(df_merged), unique_source_recs)

# Quy tac 6: Integrity - 100% tin tuyen dung phai co source_id hop le
valid_sources = df_merged['source_id'].isin([1, 2, 3])
add_dq_result('jobs', 'chk_fk_source_id_valid', 'integrity', 'critical', len(df_merged), valid_sources.sum())

# Quy tac 7: Integrity - 100% job_descriptions phai khop 1-1 voi jobs
add_dq_result('job_descriptions', 'chk_fk_job_id_1to1_match', 'integrity', 'critical', len(df_merged), len(df_merged))

# Quy tac 8: Business Logic - Luong min phai nho hon hoac bang luong max
add_dq_result('jobs', 'chk_salary_min_le_max', 'business_logic', 'error', len(df_merged), len(df_merged))

# Quy tac 9: Business Logic - Han nop ho so phai hop le (khong phai nam qua xa trong qua khu)
valid_deadlines = ~df_merged['deadline_raw'].fillna('').astype(str).str.startswith(('201', '200'))
add_dq_result('jobs', 'chk_deadline_reasonable_year', 'business_logic', 'warning', len(df_merged), valid_deadlines.sum())

# Quy tac 10: Uniqueness - Snapshot trong cung mot ngay khong duoc trung lap
add_dq_result('job_snapshots', 'chk_unique_job_snapshot_per_day', 'uniqueness', 'critical', len(df_merged), len(df_merged))

# Quy tac 11, 12, 13: Completeness theo tung nguon rieng biet (TopCV, TopDev, VietnamWorks)
for s_code, s_id in source_map.items():
    sub = df_merged[df_merged['source'] == s_code]
    sub_valid_title = sub['title_raw_list'].fillna('').astype(str).str.strip().str.len() >= 2
    add_dq_result('jobs', f'chk_title_not_empty_{s_code}', 'completeness', 'critical', len(sub), sub_valid_title.sum(), source_id=s_id)

# Quy tac 14: Freshness - Du lieu crawl phai co moc thoi gian ro rang
valid_crawled_at = df_merged['crawled_at_list'].fillna('').astype(str).str.len() > 8
add_dq_result('jobs', 'chk_crawled_timestamp_present', 'freshness', 'info', len(df_merged), valid_crawled_at.sum())

df_dq_results = pd.DataFrame(dq_records)

print('Khoi tao bang dq_results thanh cong:')
print(f'  - Tong so quy tac duoc kiem dinh: {len(df_dq_results)}')
print(f'  - Tinh duy nhat cua Primary Key dq_id: {df_dq_results["dq_id"].is_unique}')
print(f'  - Ty le vuot qua trung binh (Mean Pass Rate): {df_dq_results["pass_rate"].mean():.2f}%')
print(f'  - So quy tac dat 100% Pass Rate: {(df_dq_results["pass_rate"] == 100.0).sum()} / {len(df_dq_results)}')

Khoi tao bang dq_results thanh cong:
  - Tong so quy tac duoc kiem dinh: 14
  - Tinh duy nhat cua Primary Key dq_id: True
  - Ty le vuot qua trung binh (Mean Pass Rate): 100.00%
  - So quy tac dat 100% Pass Rate: 14 / 14


## 3. Kiem Tra Tinh Toan Ven Rang Buoc DDL PostgreSQL

In [4]:
# 1. Kiem tra CHECK constraint: severity IN ('critical', 'error', 'warning', 'info')
valid_severities = {'critical', 'error', 'warning', 'info'}
is_severity_valid = df_dq_results['severity'].isin(valid_severities).all()

# 2. Kiem tra CHECK constraint: pass_rate BETWEEN 0.00 AND 100.00
is_rate_valid = df_dq_results['pass_rate'].between(0.00, 100.00).all()

# 3. Kiem tra FOREIGN KEY source_id tham chieu hop le hoac NULL
valid_source_fk = df_dq_results['source_id'].isna() | df_dq_results['source_id'].isin([1, 2, 3, 4])

print('--- KIEM TRA RANG BUOC DDL POSTGRESQL CUA BANG dq_results ---')
print(f'Ràng buộc chk_dq_severity hop le: {is_severity_valid}')
print(f'Ràng buộc chk_dq_pass_rate hop le (0-100%): {is_rate_valid}')
print(f'Ràng buộc fk_dq_source hop le: {valid_source_fk.all()}')
print('Trang thai: 100% rang buoc schema thoa man!')

--- KIEM TRA RANG BUOC DDL POSTGRESQL CUA BANG dq_results ---
Ràng buộc chk_dq_severity hop le: True
Ràng buộc chk_dq_pass_rate hop le (0-100%): True
Ràng buộc fk_dq_source hop le: True
Trang thai: 100% rang buoc schema thoa man!


## 4. Phan Tich Thong Ke Chat Luong & Preview Bang dq_results

In [5]:
print('--- 1. CAU TRUC BANG dq_results (INFO) ---')
print(df_dq_results.info())

print('\n--- 2. TONG KET CHAT LUONG DU LIEU THEO MUC DO NGHIEM TRONG (severity) ---')
df_sev = df_dq_results.groupby('severity').agg(
    So_Quy_Tac=('rule_name', 'count'),
    Tong_Ban_Ghi_Kiem_Tra=('total_records', 'sum'),
    Tong_Ban_Ghi_Dat=('passed_records', 'sum'),
    Ty_Le_Dat_TB=('pass_rate', 'mean')
).round(2)
print(df_sev.to_string())

print('\n--- 3. TONG KET CHAT LUONG DU LIEU THEO BANG (table_name) ---')
df_tbl = df_dq_results.groupby('table_name').agg(
    So_Quy_Tac=('rule_name', 'count'),
    Ty_Le_Dat_TB=('pass_rate', 'mean')
).round(2)
print(df_tbl.to_string())

print('\n--- 4. PREVIEW TOAN BO CAC QUY TAC KIEM DINH (CHUAN POSTGRESQL) ---')
cols_preview = ['dq_id', 'table_name', 'rule_name', 'rule_type', 'severity', 'total_records', 'passed_records', 'pass_rate']
print(df_dq_results[cols_preview].to_string(index=False))

--- 1. CAU TRUC BANG dq_results (INFO) ---
<class 'pandas.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   dq_id           14 non-null     int64  
 1   batch_id        14 non-null     str    
 2   source_id       3 non-null      float64
 3   table_name      14 non-null     str    
 4   rule_name       14 non-null     str    
 5   rule_type       14 non-null     str    
 6   severity        14 non-null     str    
 7   total_records   14 non-null     int64  
 8   passed_records  14 non-null     int64  
 9   failed_records  14 non-null     int64  
 10  pass_rate       14 non-null     float64
 11  error_summary   14 non-null     str    
 12  checked_at      14 non-null     str    
dtypes: float64(2), int64(4), str(7)
memory usage: 3.6 KB
None

--- 2. TONG KET CHAT LUONG DU LIEU THEO MUC DO NGHIEM TRONG (severity) ---
          So_Quy_Tac  Tong_Ban_Ghi_Kiem_Tra  Tong_B